# Lesson 4: Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
tool = TavilySearchResults(max_results=2)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [11]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [12]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [13]:
thread = {"configurable": {"thread_id": "1"}}

In [15]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FKttOaGX6OySFcN5KodyRrpe', 'function': {'arguments': '{"query":"San Francisco current weather November 2023"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 1590, 'total_tokens': 1615, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_deacdd5f6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-fcbeb047-877b-4135-bc9b-8d0f69d9f01e-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco current weather November 2023'}, 'id': 'call_FKttOaGX6OySFcN5KodyRrpe'}])]
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco current weather November 2023'}, 'id': 'cal

In [16]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_eTvDCryqqNr5YtwaUeYb0UZT', 'function': {'arguments': '{"query":"current weather in Los Angeles"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 1720, 'total_tokens': 1742, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_deacdd5f6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-813999a6-743b-40d0-8e63-3fb759e4dee1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Los Angeles'}, 'id': 'call_eTvDCryqqNr5YtwaUeYb0UZT'}])]}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Los Angeles'}, 'id': 'call_eTvDCryqqNr5YtwaUeYb0UZ

In [17]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_naBoir2p8jLvj7gYmqr5bxsY', 'function': {'arguments': '{"query": "current temperature in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_4NMwIXxvyGc91kz3Vlxi4O1X', 'function': {'arguments': '{"query": "current temperature in Los Angeles"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 1848, 'total_tokens': 1908, 'prompt_tokens_details': {'cached_tokens': 1536, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_deacdd5f6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-b1417364-bbb5-43e4-92b7-ecc8f9d98517-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current temperature in San

{'messages': [AIMessage(content='Based on the current weather data, Los Angeles is warmer than San Francisco. Here are the details:\n\n- **San Francisco:** The current high temperature is around 62°F.\n- **Los Angeles:** The current high temperature is around 65°F.\n\nIf you have any more questions or need further assistance, feel free to ask!', response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 5542, 'total_tokens': 5609, 'prompt_tokens_details': {'cached_tokens': 1792, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_deacdd5f6f', 'finish_reason': 'stop', 'logprobs': None}, id='run-8643e36c-8610-4572-b85a-9f85c742c216-0')]}


In [18]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Could you please clarify what you're comparing to determine which is warmer? Are you comparing two specific locations, types of clothing, materials, or something else? Let me know so I can provide the appropriate information.", response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 149, 'total_tokens': 192, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_f9f4fb6dbf', 'finish_reason': 'stop', 'logprobs': None}, id='run-e5e150e8-c680-4997-a20c-701ebba67805-0')]}


## Streaming tokens

In [19]:
from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver

memory = AsyncSqliteSaver.from_conn_string(":memory:")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [21]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
kinds=[]
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    kinds.append(kind)
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather San Francisco'}, 'id': 'call_6a43b9KEyfzY5FVAFsNEqa2v'}
Back to the model!
I| was| unable| to| find| the| current| weather| for| San| Francisco| in| the| latest| search| results|.| To| get| the| most| accurate| and| up|-to|-date| weather| information|,| consider| checking| a| reliable| weather| website| or| app| like| Weather|.com| or| Acc|u|Weather|.|

In [23]:
kinds

['on_chain_start',
 'on_chain_start',
 'on_chain_end',
 'on_chain_start',
 'on_chat_model_start',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_end',
 'on_chain_start',
 'on_chain_end',
 'on_chain_start',
 'on_chain_end',
 'on_chain_start',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_start',
 'on_tool_start',
 'on_tool_end',
 'on_chain_start',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_start',
 'on_chat_model_start',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 'on_chat_model_stream',
 